In [ ]:
from fw_presidio_image_redactor.fw_scan_and_redact import FwScanRedactEngine

import easyocr
import pydicom
import pathlib
import tempfile
from PIL import Image
from pathlib import Path
from presidio_image_redactor import DicomImageRedactorEngine, ImageAnalyzerEngine, ContrastSegmentedImageEnhancer
from presidio_image_redactor.entities import ImageRecognizerResult
from typing import Tuple, List, Optional
import subprocess


In [ ]:
input_files = [Path("/flywheel/v0/docs/notebooks/test_images/davidson.dcm")]
phi_config = False 
use_meta = False

MyEngine = FwScanRedactEngine(input_files=input_files, phi_config=phi_config, use_meta=use_meta)
subprocess.run(["rm", "-rf", "separated_us_images"])


In [ ]:
class EasyOCR():
    def __init__(self,):
        self.language = ['en']

    def perform_ocr(self, file_path:str, lang, config) -> None:
        reader = easyocr.Reader(self.language)
        result = reader.readtext(file_path)
        formatted_result = self._easy_result_format(result)
        return formatted_result
    
    def _easy_result_format(self,easy_result:list) -> list: 
        left = []
        top = []
        width = []
        height = []
        text = []
        conf_score = []
        for entry in easy_result:
            bounding_boxes = entry[0]
            text.append(entry[1])
            conf_score.append(entry[2])
            left.append(bounding_boxes[3][0])
            top.append(bounding_boxes[3][1])
            width.append(bounding_boxes[2][0]-bounding_boxes[3][0])
            height.append(bounding_boxes[3][1]-bounding_boxes[0][0])
        
        formatted_output = {
            "left": left,
            "top": top,
            "width": width,
            "height": height,
            "conf": conf_score,
            "text": text
        }
        return formatted_output

In [ ]:
MyEngine.image_analyzer_engine.ocr = EasyOCR()
analyzer_results, bbox_coords, phi_found = MyEngine.scan_dicoms_for_phi(
    gen_bbox_images=True,
    output_path="/flywheel/v0/output"
)